# Day 7 Project: Long-Document Summarizer CLI

## What You're Building

You run `summarize_document(text, chunk_size=500, overlap=50, model='llama3.2')` with a long text
(e.g. a pasted article or the contents of a `.txt` file).

The function prints each chunk summary as it is produced (map phase), then prints a single
final summary (reduce phase). You see intermediate progress **and** a coherent final result
that captures the whole document — despite it being longer than the model's practical context
limit.

That's the deliverable.

---

## Concepts This Project Composes

| Step | Concept | Lesson |
|------|---------|--------|
| Measure | Context windows + token estimation | Lesson 1 |
| Split | Fixed-size chunking with overlap | Lesson 2 |
| Map | `summarize_chunk()` + system prompt | Lessons 3 & 4 |
| Reduce | `reduce_summaries()` — one final call | Lesson 5 |
| Entry point | `summarize_document()` — hides all complexity | Lesson 5 |

> **Before you start:** complete exercises 1–5 in the `exercises/` folder.
>
> **Ollama must be running:** `ollama serve` in a terminal, model pulled with `ollama pull llama3.2`.

## Step 1: Imports and Constants

Set up everything you need before writing any functions.

In [ ]:
import ollama

# --- Model ---
# Step 1a: Define the model name constant.
MODEL = "llama3.2"

# --- Context-window constants (from Lesson 1) ---
# Step 1b: Fill in the values you learned in Lesson 1.
CONTEXT_WINDOW_TOKENS = # TODO: conservative limit for llama3.2 (2048)
CHARS_PER_TOKEN       = # TODO: approximate characters per token (4)
PROMPT_OVERHEAD_TOKENS = # TODO: headroom for the system prompt (200)

MAX_DOC_TOKENS = CONTEXT_WINDOW_TOKENS - PROMPT_OVERHEAD_TOKENS
MAX_DOC_CHARS  = MAX_DOC_TOKENS * CHARS_PER_TOKEN

# --- System prompts ---
# Step 1c: Write the map-phase system prompt (2-3 sentences, plain prose, no additions).
SUMMARIZE_SYSTEM_PROMPT = (
    # TODO
)

# Step 1d: Write the reduce-phase system prompt (synthesise multiple summaries into one).
REDUCE_SYSTEM_PROMPT = (
    # TODO
)

## Step 2: Measurement Helpers

Implement the two functions from Lesson 1 that tell you whether a document needs splitting.

In [ ]:
def estimate_tokens(text: str) -> int:
    """
    Return a rough token count for English prose.
    Uses the 1 token ≈ 4 characters approximation.
    """
    # TODO
    pass


def will_fit(text: str) -> bool:
    """
    Return True if the document is likely within the model's context window.
    """
    # TODO
    pass

## Step 3: Chunking

Implement `chunk_text` from Lesson 2. Remember: `step = chunk_size - overlap`; the cursor advances
by `step`, not by `chunk_size`.

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into fixed-size chunks with a shared overlap region.

    Args:
        text:       The document to split.
        chunk_size: Maximum number of characters per chunk.
        overlap:    Characters shared between consecutive chunks.
                    Must be strictly less than chunk_size.

    Returns:
        A list of non-empty strings. Returns [] when text is empty.
    """
    # Step 3a: Guard: return [] for empty text.
    # TODO

    # Step 3b: Guard: raise ValueError if overlap >= chunk_size.
    # TODO

    # Step 3c: Compute step, initialise cursor and output list.
    # TODO

    # Step 3d: While cursor < len(text): slice, append, advance.
    # TODO

    pass

## Step 4: Map Phase — Summarize One Chunk

Implement `summarize_chunk` (Lesson 3): one string in, one summary string out.

In [ ]:
def summarize_chunk(chunk: str, model: str = MODEL) -> str:
    """
    Summarize a single text chunk using a local Ollama model.

    Args:
        chunk: A string that fits within the model's context window.
        model: Ollama model name.

    Returns:
        A plain-prose summary string (2–3 sentences).
    """
    # Step 4a: Build the messages list — system prompt first, chunk as user message.
    # TODO

    # Step 4b: Call ollama.chat(model=model, messages=messages).
    # TODO

    # Step 4c: Return response["message"]["content"].
    # TODO
    pass

## Step 5: Reduce Phase — Combine All Chunk Summaries

Implement `reduce_summaries` (Lesson 5): join the intermediate summaries and make one final
synthesis call.

In [ ]:
def reduce_summaries(chunk_summaries: list[str], model: str = MODEL) -> str:
    """
    Combine a list of per-chunk summaries into one final summary.

    Args:
        chunk_summaries: List of short summary strings from the map phase.
        model:           Ollama model name.

    Returns:
        A single coherent summary of the whole document.
    """
    # Step 5a: Join chunk_summaries with "\n\n" as separator.
    # TODO

    # Step 5b: Build the messages list — REDUCE_SYSTEM_PROMPT first, combined text as user.
    # TODO

    # Step 5c: Call ollama.chat once and return the content string.
    # TODO
    pass

## Step 6: Wire It Together — `summarize_document`

The entry point. It hides all Map-Reduce complexity behind a single call.
Print each chunk summary as it is produced (map phase), then print the final summary.

In [ ]:
def summarize_document(
    text: str,
    chunk_size: int = 500,
    overlap: int = 50,
    model: str = MODEL,
) -> str:
    """
    Summarize a document of any length using Map-Reduce.

    Short documents (fit in one context window) go directly to the model.
    Long documents are chunked → each chunk is summarised (map) → all
    chunk summaries are synthesised into one final result (reduce).

    Prints each chunk summary as it is produced, then prints the final summary.

    Args:
        text:       The full document text to summarise.
        chunk_size: Maximum characters per chunk.
        overlap:    Characters shared between consecutive chunks.
        model:      Ollama model name.

    Returns:
        A single summary string covering the entire document.
    """
    # Step 6a: If will_fit(text), summarize_chunk the whole doc and return early.
    # TODO

    # Step 6b: Otherwise, split with chunk_text(text, chunk_size, overlap).
    # TODO

    # Step 6c: Map phase — loop over chunks, call summarize_chunk on each,
    #          print a progress line and the chunk summary as you go.
    # TODO

    # Step 6d: Reduce phase — call reduce_summaries, print and return the result.
    # TODO
    pass

## Step 7: Run It

Paste a long article, or use the sample text below (a repeated paragraph that
forces chunking). Watch the map phase print each chunk summary, then the reduce
phase print one final answer.

In [ ]:
# --- Sample long document (forces the map-reduce path) ---
SAMPLE_TEXT = (
    "Global temperatures have risen steadily over the past century due to "
    "greenhouse gas emissions from industrial activity and deforestation. "
    "Scientists have documented melting ice caps, rising sea levels, and "
    "increasingly severe weather events as consequences of this warming trend. "
) * 28  # ~7 800 chars — above the 7 392-char context limit, forces map-reduce

# --- Run the pipeline ---
# Replace SAMPLE_TEXT with any long string you like.
final_summary = summarize_document(
    SAMPLE_TEXT,
    chunk_size=2500,   # ~3-4 chunks → manageable number of map calls
    overlap=100,
    model="llama3.2",
)

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(final_summary)